# Example - Reading and Writing with Dask

In [1]:
import multiprocessing
# Linux/OSX:
import multiprocessing.popen_spawn_posix
# Windows:
# import multiprocessing.popen_spawn_win32
import threading

from dask.distributed import Client, LocalCluster, Lock

import rioxarray

Tips for using dask locks:
- Be careful about what lock you use for your process. It is required to have a lock for each worker, so the more fine-grained the better.
- The reading and writing processes need the same type of lock. They don't have to share the same lock, but they do nead a lock of the same type.

See docs for:

- [Reading COGs in Parallel](read-locks.ipynb)
- [rioxarray.open_rasterio](../rioxarray.rst#rioxarray-open-rasterio)
- DataArray: [rio.to_raster()](../rioxarray.rst#rioxarray.raster_array.RasterArray.to_raster)
- Dataset: [rio.to_raster()](../rioxarray.rst#rioxarray.raster_dataset.RasterDataset.to_raster)

### No distributed computing example
Note: Without a lock provided, `to_raster` does not use dask to write to disk.

In [2]:
xds = rioxarray.open_rasterio(
    "../../test/test_data/compare/small_dem_3m_merged.tif",
    chunks=True,
)
xds.rio.to_raster("simple_write.tif", tiled=True)

### Deferring writes with `compute=False`

Deferred writing requires **both** a Dask-backed array and a lock (`True` or a
lock object). Only then does `compute=False` return a Dask collection of write
tasks; the exact collection type depends on Dask. With `compute=True` the writes
finish immediately and `None` is returned. Without Dask-backed data or without
a lock, the call writes synchronously and returns `None` even with `compute=False`.
This also applies to `Dataset.rio.to_raster`.

`compute=False` does **not** prevent file creation or overwrite: the output file
and its metadata are created immediately; writing the array data is deferred.
Use a suitable lock for your scheduler. This small, self-contained example uses
a local synchronous scheduler and removes its two temporary output files:


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import dask
from dask.base import is_dask_collection
from dask.utils import SerializableLock
import numpy as np
import rasterio
import xarray as xr

import rioxarray  # registers the rio accessor

pixels = np.arange(1, 13, dtype=np.int16).reshape(3, 4)
raster = xr.DataArray(
    pixels,
    coords={"y": [10.0, 9.0, 8.0], "x": [10.0, 11.0, 12.0, 13.0]},
    dims=("y", "x"),
).rio.write_crs("EPSG:4326").chunk({"y": 1, "x": 2})

with TemporaryDirectory() as output_directory:
    paths = [Path(output_directory) / name for name in ("first.tif", "second.tif")]
    lock = SerializableLock()
    writes = [raster.rio.to_raster(path, lock=lock, compute=False) for path in paths]
    assert all(is_dask_collection(write) for write in writes)
    # Files and metadata already exist; array data has not been written yet.
    assert all(path.exists() for path in paths)
    with dask.config.set(scheduler="synchronous"):
        dask.compute(*writes)
    for path in paths:
        with rasterio.open(path) as written:
            np.testing.assert_array_equal(written.read(1), pixels)


### Multithreaded example

In [3]:
xds = rioxarray.open_rasterio(
    "../../test/test_data/compare/small_dem_3m_merged.tif",
    chunks=True,
    lock=False,
    # lock=threading.Lock(), # when too many file handles open
xds.rio.to_raster(
    "dask_thread.tif", tiled=True, lock=threading.Lock(),
)

### Multiple worker example

In [4]:
with LocalCluster() as cluster, Client(cluster) as client:
    xds = rioxarray.open_rasterio(
        "../../test/test_data/compare/small_dem_3m_merged.tif",
        chunks=True,
        lock=False,
        # lock=Lock("rio-read", client=client), # when too many file handles open
    )
    xds.rio.to_raster(
        "dask_multiworker.tif",
        tiled=True,
        lock=Lock("rio", client=client),
    )